In [0]:
%sql
DROP TABLE IF EXISTS bronze.employees;
DROP TABLE IF EXISTS bronze.salaries;
DROP TABLE IF EXISTS bronze.departments;

In [0]:
%sql
USE CATALOG hr_catalog_divansh;
USE SCHEMA bronze;

In [0]:
%sql
SELECT current_catalog(), current_schema();

In [0]:
%sql
SHOW TABLES;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# -----------------------------
# 1. Define Schemas (IMPORTANT)
# -----------------------------

employee_schema = StructType([
    StructField("employee_id", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("department_id", StringType(), True),
    StructField("job_title", StringType(), True),
    StructField("hire_date", StringType(), True),
    StructField("employment_type", StringType(), True),
    StructField("manager_id", StringType(), True)
])

salary_schema = StructType([
    StructField("salary_id", StringType(), True),
    StructField("employee_id", StringType(), True),
    StructField("base_salary", DoubleType(), True),
    StructField("bonus_pct", DoubleType(), True),
    StructField("effective_date", StringType(), True),
    StructField("currency", StringType(), True)
])

department_schema = StructType([
    StructField("department_id", StringType(), True),
    StructField("department_name", StringType(), True),
    StructField("location", StringType(), True),
    StructField("head_count_budget", IntegerType(), True)  # FIXED
])

# -----------------------------
# 2. Read CSV Files
# -----------------------------

employees_raw = spark.read \
    .option("header", True) \
    .schema(employee_schema) \
    .csv("/FileStore/hr_divansh/raw/employees.csv")

salaries_raw = spark.read \
    .option("header", True) \
    .schema(salary_schema) \
    .csv("/FileStore/hr_divansh/raw/salaries.csv")

departments_raw = spark.read \
    .option("header", True) \
    .schema(department_schema) \
    .csv("/FileStore/hr_divansh/raw/departments.csv")

# -----------------------------
# 3. Add Metadata Columns
# -----------------------------

def add_metadata(df):
    return df \
        .withColumn("_ingested_at", F.current_timestamp()) \
        .withColumn("_source_file", F.expr("_metadata.file_path"))

employees_df = add_metadata(employees_raw)
salaries_df = add_metadata(salaries_raw)
departments_df = add_metadata(departments_raw)

# -----------------------------
# 4. Write Bronze Tables
# -----------------------------

employees_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hr_catalog_divansh.bronze.employees")

salaries_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hr_catalog_divansh.bronze.salaries")

departments_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hr_catalog_divansh.bronze.departments")

# -----------------------------
# 5. Add CHECK Constraint
# -----------------------------

spark.sql("""
ALTER TABLE hr_catalog_divansh.bronze.employees
ADD CONSTRAINT valid_employment_type
CHECK (employment_type IN ('full_time', 'part_time', 'contract'))
""")

# -----------------------------
# 6. Verify Delta Log
# -----------------------------

spark.sql("DESCRIBE HISTORY bronze.employees").show(truncate=False)

# -----------------------------
# 7. Time Travel Check
# -----------------------------

spark.sql("SELECT * FROM bronze.employees VERSION AS OF 0").show(5)

# -----------------------------
# 8. Auto Loader (Employee Updates)
# -----------------------------

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/FileStore/hr_divansh/schema/employees_updates")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", True)
    .option("pathGlobFilter", "employees_update*.csv")
    .load("/FileStore/hr_divansh/raw/")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.expr("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", "/FileStore/hr_divansh/checkpoints/employees_updates")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("bronze.employees_updates")
)